# Parcel Analytics Workspace Smoke Test

This notebook validates that the local geospatial stack is available and shows a minimal parcel-style workflow.

In [1]:
import geopandas as gpd
import pandas as pd
import folium
from shapely.geometry import Polygon

import parcel_analytics

parcel_analytics.__version__

'0.1.0'

In [2]:
parcels = gpd.GeoDataFrame(
    {
        "parcel_id": ["P-001", "P-002", "P-003"],
        "land_use": ["residential", "mixed_use", "residential"],
        "site_value": [880000, 1260000, 940000],
    },
    geometry=[
        Polygon([(144.9620, -37.8136), (144.9620, -37.8129), (144.9628, -37.8129), (144.9628, -37.8136)]),
        Polygon([(144.9630, -37.8135), (144.9630, -37.8128), (144.9638, -37.8128), (144.9638, -37.8135)]),
        Polygon([(144.9640, -37.8134), (144.9640, -37.8127), (144.9648, -37.8127), (144.9648, -37.8134)]),
    ],
    crs="EPSG:4326",
)

projected = parcels.to_crs(3857)
centroids = projected.centroid.to_crs(4326)

parcels = parcels.assign(
    area_sqm=projected.area.round(1),
    centroid_lon=centroids.x.round(6),
    centroid_lat=centroids.y.round(6),
)

parcels

,parcel_id,land_use,site_value,geometry,area_sqm,centroid_lon,centroid_lat
0,P-001,residential,880000,"POLYGON ((144.962 -37.8136, 144.962 -37.8129, ...",8784.1,144.9624,-37.81325
1,P-002,mixed_use,1260000,"POLYGON ((144.963 -37.8135, 144.963 -37.8128, ...",8784.1,144.9634,-37.81315
2,P-003,residential,940000,"POLYGON ((144.964 -37.8134, 144.964 -37.8127, ...",8784.1,144.9644,-37.81305


In [3]:
summary = (
    parcels[["land_use", "site_value", "area_sqm"]]
    .groupby("land_use", as_index=False)
    .agg(total_site_value=("site_value", "sum"), mean_area_sqm=("area_sqm", "mean"))
)

summary

,land_use,total_site_value,mean_area_sqm
0,mixed_use,1260000,8784.1
1,residential,1820000,8784.1


In [4]:
parcel_map = folium.Map(location=[-37.8131, 144.9634], zoom_start=18, tiles="CartoDB positron")

for _, row in parcels.iterrows():
    folium.GeoJson(
        row.geometry,
        tooltip=f"{row.parcel_id}: ${row.site_value:,.0f}",
        style_function=lambda _: {"color": "#005f73", "weight": 2, "fillColor": "#94d2bd", "fillOpacity": 0.45},
    ).add_to(parcel_map)

parcel_map